# Epithelial sub-cluster + morphology in napari

Same setup as `07_napari_cell_type_selection.ipynb` (morphology image + cell boundary polygons + checklist dock widget + hover tooltip + minimap), but colored by the lightweight `cell_id` + `sub_leiden` export from `05_subclustering_epithelial.ipynb` instead of `cell_type`. Non-epithelial cells (absent from the export) fall into an `unknown` category.

In [1]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"


import napari
import spatialdata as sd
from shapely import from_wkb
from shapely.geometry import MultiPolygon, Polygon
from shapely.validation import make_valid
from magicgui.widgets import CheckBox, Container

In [2]:
PROJECT = Path.home() / "Projects/xenium_lung"

VIS_DIR = PROJECT / "results/xenium_5k/visualization"
RAW_DIR = PROJECT / "data/bundle/xenium_prime_5k"

COORDINATES_FILE = VIS_DIR / "cell_coordinates.parquet"
BOUNDARIES_FILE = VIS_DIR / "cell_boundaries.parquet"

# lightweight per-compartment export from 05_subclustering_epithelial.ipynb (cell_id + sub_leiden + sub_cell_type)
CLUSTERS_FILE = PROJECT / "results/xenium_5k/annotation/subclustering/epithelial/sub_cluster_labels.csv"
COLOR_COL = "sub_leiden"  # switch to "sub_cell_type" once subcluster_labels has been filled in

# morphology.ome.tif is the raw unprojected DAPI z-stack; morphology_focus/*.ome.tif are the
# focus-projected, multi-channel images (same channels referenced across all 4 files via OME metadata).
MORPHOLOGY_FILE = RAW_DIR / "morphology_focus" / "morphology_focus_0000.ome.tif"

# H&E image + alignment only — same SpatialData store used in 06/07
SDATA_PATH = PROJECT / "data/spatialdata/xenium_prime_5k_processed_annotated.zarr"
HE_ALIGN_PATH = RAW_DIR / "Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv"


print("Clusters:", CLUSTERS_FILE)
print("Exists:", CLUSTERS_FILE.exists())
print("Morphology:", MORPHOLOGY_FILE)
print("Exists:", MORPHOLOGY_FILE.exists())
print("H&E alignment:", HE_ALIGN_PATH)
print("Exists:", HE_ALIGN_PATH.exists())

print("SpatialData:", SDATA_PATH)
print("Exists:", SDATA_PATH.exists())

Clusters: /home/duydao/Projects/xenium_lung/results/xenium_5k/annotation/subclustering/epithelial/sub_cluster_labels.csv
Exists: True
Morphology: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k/morphology_focus/morphology_focus_0000.ome.tif
Exists: True
H&E alignment: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k/Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv
Exists: True
SpatialData: /home/duydao/Projects/xenium_lung/data/spatialdata/xenium_prime_5k_processed_annotated.zarr
Exists: True


# Load the cell information

In [17]:
coordinates = pd.read_parquet(COORDINATES_FILE)
clusters = pd.read_csv(CLUSTERS_FILE)[["cell_id", COLOR_COL]].rename(columns={COLOR_COL: "group"})

cells = coordinates.merge(
    clusters,
    on="cell_id",
    how="left",
    validate="one_to_one"
)

print("Cells:", cells.shape)
print(cells.head())

Cells: (266179, 4)
      cell_id           x            y  group
0  aaaaadnb-1  822.469055  5111.537598    NaN
1  aaaabalp-1  843.901428  5149.261230    NaN
2  aaaadfei-1  831.219421  5133.179199    NaN
3  aaaadjia-1  839.742981  5159.693848    NaN
4  aaaafglb-1  784.250916  5143.785645    NaN


In [19]:
import scanpy as sc
from matplotlib.colors import to_rgba

# Sort numerically ("0", "1", ..., "10") to match the sub_leiden category order from 05_subclustering_epithelial.ipynb
cluster_ids = sorted(
    cells["group"].dropna().astype(str).unique(),
    key=lambda c: int(float(c)),
)

# Same categorical palette scanpy assigns to sub_leiden in 05_subclustering_epithelial.ipynb
palette = sc.pl.palettes.default_20 if len(cluster_ids) <= 20 else sc.pl.palettes.default_102
color_cycle = [to_rgba(palette[i]) for i in range(len(cluster_ids))]

cells["group"] = cells["group"].astype("object")
cells.loc[cells["group"].notna(), "group"] = cells.loc[cells["group"].notna(), "group"].astype(str)
cells["group"] = cells["group"].fillna("unknown")

shape_categories = cluster_ids
shape_color_cycle = color_cycle

print("Clusters:", cluster_ids)

Clusters: ['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0', '10.0', '11.0']


In [5]:
import re
import tifffile

with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    ome_xml = tif.ome_metadata

# cell x/y are in microns; the TIFF pixel grid needs this scale to line up with them
match_x = re.search(r'PhysicalSizeX="([\d.]+)"', ome_xml)
match_y = re.search(r'PhysicalSizeY="([\d.]+)"', ome_xml)

pixel_size_x = float(match_x.group(1)) if match_x else 1.0
pixel_size_y = float(match_y.group(1)) if match_y else 1.0

channel_names = re.findall(r'<Channel[^>]*Name="([^"]+)"', ome_xml)

print("Channel names:", channel_names)
print("Pixel size (x, y):", pixel_size_x, pixel_size_y)

Channel names: ['DAPI', 'ATP1A1/CD45/E-Cadherin', '18S', 'alphaSMA/Vimentin']
Pixel size (x, y): 0.2125 0.2125


# Open napari

In [6]:
viewer = napari.Viewer()

/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),
/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


# Add the H&E image

In [7]:
sdata = sd.read_zarr(SDATA_PATH)

he_image = sdata.images["he_image"]
he_scale_keys = sorted(he_image.keys())
he_pyramid = [
    he_image[key]["image"].transpose("y", "x", "c").data
    for key in he_scale_keys
]

# Alignment CSV is a (x, y, 1) affine mapping full-res H&E pixels -> global (morphology) pixel space.
he_affine_xy = pd.read_csv(HE_ALIGN_PATH, header=None).to_numpy()

a, b, tx = he_affine_xy[0]
c, d, ty = he_affine_xy[1]

he_affine_yx = np.array(
    [
        [d, c, ty],
        [b, a, tx],
        [0, 0, 1],
    ]
)

# The morphology/boundary layers use a plain pixel->micron scale factor, not a baked-in affine;
# fold the same pixel size into the H&E affine so it lines up in the same micron space.
he_affine_um = he_affine_yx.copy()
he_affine_um[0, :] *= pixel_size_y
he_affine_um[1, :] *= pixel_size_x

viewer.add_image(
    he_pyramid,
    multiscale=True,
    affine=he_affine_um,
    rgb=True,
    units=("um", "um"),
    name="H&E",
)

he_shape = he_pyramid[0].shape  # (y, x, c)
he_corners_yx = np.array([[0, 0], [0, he_shape[1]], [he_shape[0], 0], [he_shape[0], he_shape[1]]])
he_corners_world = (he_affine_um @ np.column_stack([he_corners_yx, np.ones(4)]).T).T[:, :2]
print("H&E world bbox (y, x):", he_corners_world.min(axis=0), "to", he_corners_world.max(axis=0))

/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
no parent found for <ome_zarr.reader.Label object at 0x7391b59d9850>: None
no parent found for <ome_zarr.reader.Label object at 0x7391b59ad6a0>: None


H&E world bbox (y, x): [  17.06654657 -160.60478634] to [ 7422.73001522 11740.92670632]


# Load morphology image

The morphology TIFF is a multiscale (pyramidal) OME-TIFF — reading the full series would require 64 GiB of RAM. Instead build a list of dask arrays, one per resolution level, so only visible tiles are read from disk.

In [8]:
import dask.array as da
import zarr

# Read the whole OME-TIFF pyramid as one multiscale zarr group (keys "0", "1", ... = levels).
# This is more reliable than iterating tif.series[0].levels, which duplicated/omitted levels here.
store = tifffile.imread(MORPHOLOGY_FILE, aszarr=True)
group = zarr.open(store, mode="r")
level_keys = sorted(group.array_keys(), key=int)

pyramid = [da.from_array(group[key], chunks="auto") for key in level_keys]

for i, level in enumerate(pyramid):
    print(i, "shape =", level.shape, "dtype =", level.dtype)

0 shape = (4, 37348, 54086) dtype = uint16
1 shape = (4, 18674, 27043) dtype = uint16
2 shape = (4, 9337, 13521) dtype = uint16
3 shape = (4, 4668, 6760) dtype = uint16
4 shape = (4, 2334, 3380) dtype = uint16
5 shape = (4, 1167, 1690) dtype = uint16
6 shape = (4, 583, 845) dtype = uint16
7 shape = (4, 291, 422) dtype = uint16


In [9]:
for name in ["Morphology"] + channel_names:
    if name in viewer.layers:
        viewer.layers.remove(name)

viewer.add_image(
    pyramid,
    channel_axis=0,
    name=channel_names,
    multiscale=True,
    contrast_limits=[0, 20000],
    scale=(pixel_size_y, pixel_size_x),
    units=("um", "um"),
    visible=False,
)

morph_shape = pyramid[0].shape
print("Morphology world bbox (y, x): [0, 0] to", (morph_shape[-2] * pixel_size_y, morph_shape[-1] * pixel_size_x))

Morphology world bbox (y, x): [0, 0] to (7936.45, 11493.275)


# Build cell boundary polygons for cells with a cluster label (filtering is done via layer visibility, not by rebuilding)

Only cells with a real `group` (`sub_leiden`) value are loaded — cells absent from the sub-clustering export are skipped. The selector below toggles visibility per `group` without rebuilding the layer.

In [20]:
cluster_lookup = cells.set_index("cell_id")["group"]
cluster_lookup = cluster_lookup[cluster_lookup != "unknown"]

boundaries = pd.read_parquet(BOUNDARIES_FILE)
boundaries = boundaries[boundaries.index.astype(str).isin(cluster_lookup.index.astype(str))]
geometries = from_wkb(boundaries["geometry"].to_numpy())

polygon_data = []
polygon_groups = []
n_skipped = 0

for cell_id, geom in zip(boundaries.index, geometries):
    # Self-intersecting/degenerate polygons crash vispy's triangulator — repair or drop them.
    if not geom.is_valid:
        geom = make_valid(geom)

    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon) or geom.is_empty:
        n_skipped += 1
        continue

    coords = np.asarray(geom.exterior.coords)

    keep = np.ones(len(coords), dtype=bool)
    keep[1:] = np.any(np.diff(coords, axis=0) != 0, axis=1)
    coords = coords[keep]
    if len(coords) < 4:  # closed ring needs >= 3 unique vertices + repeated first point
        n_skipped += 1
        continue

    # Shapely X,Y -> napari Y,X
    coords_yx = coords[:, [1, 0]]

    polygon_data.append(coords_yx)
    polygon_groups.append(cluster_lookup.get(str(cell_id)))

polygon_groups = np.array(polygon_groups)

print("Polygons:", len(polygon_data))
print("Skipped (invalid/degenerate):", n_skipped)

all_coords = np.concatenate(polygon_data)
print("Boundaries world bbox (y, x):", all_coords.min(axis=0), "to", all_coords.max(axis=0))

Polygons: 68416
Skipped (invalid/degenerate): 33
Boundaries world bbox (y, x): [171.48750305   6.80000019] to [ 7298.10009766 11468.41308594]


# Add cell boundary polygons filled by `group`

In [21]:
for layer in [l for l in viewer.layers if l.name.startswith("Cell boundaries")]:
    viewer.layers.remove(layer)

shapes_layer = viewer.add_shapes(
    polygon_data,
    shape_type="polygon",
    properties={"group": pd.Categorical(polygon_groups, categories=shape_categories)},
    edge_width=0,
    face_color="group",
    face_color_cycle=shape_color_cycle,
    units=("um", "um"),
    name=f"Cell boundaries — {len(polygon_data):,}",
    visible=True,
)

# Cluster of interest selector

A checklist dock widget: only the checked `group`s remain visible in the boundaries layer, via a direct `face_color` update (no rebuilding of the layer).

In [22]:
category_rgba = {category: np.array(rgba) for category, rgba in zip(shape_categories, shape_color_cycle)}
base_face_color = np.array([category_rgba[g] for g in polygon_groups])


def _to_hex(rgba):
    r, g, b = (int(round(c * 255)) for c in rgba[:3])
    return f"#{r:02x}{g:02x}{b:02x}"


checkboxes = [CheckBox(name=category, text=category, value=False) for category in shape_categories]
for cb, category in zip(checkboxes, shape_categories):
    color_hex = _to_hex(category_rgba[category])
    cb.native.setStyleSheet(
        f"QCheckBox::indicator {{ background-color: {color_hex}; border: 1px solid #888; }}"
    )


def _update_shown(event=None):
    selected = {cb.name for cb in checkboxes if cb.value}
    hidden = ~np.isin(polygon_groups, list(selected))

    new_face_color = base_face_color.copy()
    new_face_color[hidden, 3] = 0.0

    shapes_layer.face_color = new_face_color
    shapes_layer.refresh()


for cb in checkboxes:
    cb.changed.connect(_update_shown)

cluster_container = Container(widgets=checkboxes, labels=False, label="Clusters")
viewer.window.add_dock_widget(cluster_container, area="right", name="Cluster of interest")
_update_shown()

## Hover tooltip showing `group`

In [23]:
viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 12

# get_value() on this many polygons is expensive; only run it a few times per second.
TOOLTIP_INTERVAL_S = 0.08
_last_tooltip_time = 0.0


@shapes_layer.mouse_move_callbacks.append
def show_group_tooltip(layer, event):
    global _last_tooltip_time
    now = time.monotonic()
    if now - _last_tooltip_time < TOOLTIP_INTERVAL_S:
        return
    _last_tooltip_time = now

    shape_index = layer.get_value(
        event.position,
        view_direction=event.view_direction,
        dims_displayed=event.dims_displayed,
        world=True,
    )
    if isinstance(shape_index, tuple):
        shape_index = shape_index[0]

    if shape_index is not None:
        group = layer.properties["group"][shape_index]
        viewer.text_overlay.text = f"group: {group}"
    else:
        viewer.text_overlay.text = ""

## Overview / minimap panel (like Xenium Explorer's bottom-left thumbnail)

Downsampled thumbnail of the whole sample with a rectangle tracking the current camera view.

In [14]:
import matplotlib
matplotlib.use("QtAgg")
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from matplotlib.transforms import Affine2D
from qtpy.QtCore import QTimer

# Re-running this cell must not stack duplicate dock widgets / camera callbacks.
if "_overview_update_cb" in globals():
    viewer.scene.camera.events.center.disconnect(_overview_update_cb)
    viewer.scene.camera.events.zoom.disconnect(_overview_update_cb)
if "overview_dock_widget" in globals():
    try:
        viewer.window.remove_dock_widget(overview_dock_widget)
    except RuntimeError:
        pass

# Coarsest H&E pyramid level as the overview thumbnail, positioned with the same affine as the full-res layer.
he_thumb = np.asarray(he_pyramid[-1])
he_thumb_downsample_y = he_pyramid[0].shape[0] / he_pyramid[-1].shape[0]
he_thumb_downsample_x = he_pyramid[0].shape[1] / he_pyramid[-1].shape[1]
he_thumb_affine = he_affine_um @ np.diag([he_thumb_downsample_y, he_thumb_downsample_x, 1])

he_thumb_corners_yx = np.array(
    [[0, 0], [0, he_thumb.shape[1]], [he_thumb.shape[0], 0], [he_thumb.shape[0], he_thumb.shape[1]]]
)
he_thumb_corners_world = (he_thumb_affine @ np.column_stack([he_thumb_corners_yx, np.ones(4)]).T).T[:, :2]
world_y_min, world_x_min = he_thumb_corners_world.min(axis=0)
world_y_max, world_x_max = he_thumb_corners_world.max(axis=0)

# matplotlib affines act on (x, y); he_thumb_affine acts on (y, x) — swap rows/cols to convert.
A = he_thumb_affine
mpl_affine = np.array(
    [
        [A[1, 1], A[1, 0], A[1, 2]],
        [A[0, 1], A[0, 0], A[0, 2]],
        [0, 0, 1],
    ]
)

overview_fig = Figure(figsize=(3, 3))
overview_ax = overview_fig.add_subplot(111)
overview_image = overview_ax.imshow(he_thumb, extent=[0, he_thumb.shape[1], he_thumb.shape[0], 0])
overview_image.set_transform(Affine2D(mpl_affine) + overview_ax.transData)
overview_ax.set_xlim(world_x_min, world_x_max)
overview_ax.set_ylim(world_y_max, world_y_min)
overview_ax.set_xticks([])
overview_ax.set_yticks([])
overview_fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

view_rect = Rectangle((0, 0), 1, 1, edgecolor="yellow", facecolor="none", linewidth=1.5)
overview_ax.add_patch(view_rect)

overview_canvas = FigureCanvasQTAgg(overview_fig)
overview_canvas.draw()


def get_canvas_size():
    qt_viewer = getattr(viewer.window, "_qt_viewer", None) or viewer.window.qt_viewer
    size = qt_viewer.canvas.size
    return size[0], size[1]


def _redraw_overview():
    overview_canvas.draw_idle()


# Camera events fire continuously while panning/zooming; debounce with a single-shot Qt timer
# so we redraw at most a few times per second instead of on every event.
_overview_timer = QTimer()
_overview_timer.setSingleShot(True)
_overview_timer.setInterval(30)
_overview_timer.timeout.connect(_redraw_overview)


def _overview_update_cb(event=None):
    camera = viewer.scene.camera
    width_px, height_px = get_canvas_size()
    zoom = camera.zoom
    center_y, center_x = camera.center[-2:]

    half_width = width_px / zoom / 2
    half_height = height_px / zoom / 2

    view_rect.set_xy((center_x - half_width, center_y - half_height))
    view_rect.set_width(2 * half_width)
    view_rect.set_height(2 * half_height)

    _overview_timer.start()


viewer.scene.camera.events.center.connect(_overview_update_cb)
viewer.scene.camera.events.zoom.connect(_overview_update_cb)
_overview_update_cb()
_redraw_overview()

overview_dock_widget = viewer.window.add_dock_widget(overview_canvas, area="left", name="Overview")

In [15]:
viewer.scale_bar.visible = True